# 04 — Edge deployment: ONNX export and latency benchmarking

A corridor overwatch sensor is field hardware, not a data-center GPU. This notebook
exports this project's detector to ONNX (fp32 and dynamic int8 quantization) and measures
inference latency for PyTorch eager and ONNX Runtime, on CPU and (if available) GPU. See
`docs/METHODOLOGY_AND_LIMITATIONS.md#edge-deployment`.

Requires the `onnx` extra: `pip install -e ".[onnx]"`.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

Project root: C:\dev\infrastructure-overwatch


In [2]:
import torch
from torch.utils.data import DataLoader

from infrastructure_overwatch.synthetic import GRID, IMG_SIZE, N_SYNTHETIC_CLASSES, SyntheticCorridorDataset
from infrastructure_overwatch.detectors.grid_cnn import GridDetector, train_grid_detector
from infrastructure_overwatch.export import benchmark_onnx, benchmark_torch, export_onnx

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
day_train_ds = SyntheticCorridorDataset(400, domain="day", seed=1)
loader = DataLoader(day_train_ds, batch_size=32, shuffle=True)
model = GridDetector(n_classes=N_SYNTHETIC_CLASSES, grid_h=GRID, grid_w=GRID)
train_grid_detector(model, loader, epochs=10, device=device, quiet=True)
print("Trained a quick model for benchmarking (accuracy isn't the point of this notebook).")

Trained a quick model for benchmarking (accuracy isn't the point of this notebook).


In [3]:
input_shape = (1, IMG_SIZE, IMG_SIZE)
exported = export_onnx(model, input_shape, out_dir="../outputs/onnx_artifacts", name="corridor_detector")
print(f"fp32: {exported.fp32_size_kb:.1f} KB   int8: {exported.int8_size_kb:.1f} KB "
      f"({100*(1 - exported.int8_size_kb/exported.fp32_size_kb):.0f}% smaller)")

fp32: 241.6 KB   int8: 72.3 KB (70% smaller)


In [4]:
torch_cpu_latency = benchmark_torch(model, input_shape, device=torch.device("cpu"))
onnx_fp32_latency = benchmark_onnx(exported.fp32_path, input_shape)
onnx_int8_latency = benchmark_onnx(exported.int8_path, input_shape)

print(f"PyTorch eager (CPU):     {torch_cpu_latency:.3f} ms/inference")
print(f"ONNX Runtime fp32 (CPU): {onnx_fp32_latency:.3f} ms/inference")
print(f"ONNX Runtime int8 (CPU): {onnx_int8_latency:.3f} ms/inference")

if torch.cuda.is_available():
    torch_gpu_latency = benchmark_torch(model, input_shape, device=torch.device("cuda"))
    print(f"PyTorch eager (GPU, {torch.cuda.get_device_name(0)}): {torch_gpu_latency:.3f} ms/inference")
    import onnxruntime as ort
    if "CUDAExecutionProvider" in ort.get_available_providers():
        onnx_gpu_latency = benchmark_onnx(exported.fp32_path, input_shape, provider="CUDAExecutionProvider")
        print(f"ONNX Runtime fp32 (GPU): {onnx_gpu_latency:.3f} ms/inference")
    else:
        print("onnxruntime has no CUDAExecutionProvider installed -- GPU ONNX latency not measured.")
else:
    print("No CUDA GPU available on this machine -- GPU latency not measured.")

PyTorch eager (CPU):     1.514 ms/inference
ONNX Runtime fp32 (CPU): 0.236 ms/inference
ONNX Runtime int8 (CPU): 3.774 ms/inference
No CUDA GPU available on this machine -- GPU latency not measured.


## Takeaway

Read the int8-vs-fp32 numbers above the way `docs/METHODOLOGY_AND_LIMITATIONS.md` insists
on: quantization's storage win is close to guaranteed, its *latency* win depends on model
scale and this runtime's kernel support on this hardware -- report what was actually
measured here, not the generic "quantization is faster" assumption.